# Under Observation: reproduce the analysis

Individual analysis: Maitreyee Singh. Shared MARA project lead: Shreya Mendi; PI: Dr. Bent, TRUST Lab.

This notebook uses saved data and local statistical models. No Qwen API key or model download is required. In Jupyter, place the notebook inside the repository. In Colab, upload the complete repository ZIP when prompted.

In [ ]:
from pathlib import Path
import json, sys, subprocess, tempfile, zipfile
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p/'data/source/game_metrics.csv').exists()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Run this notebook from the repository or its notebooks folder.')
    uploaded = files.upload()
    archive = next((Path(name) for name in uploaded if zipfile.is_zipfile(name)), None)
    if archive is None:
        raise RuntimeError('Upload the complete project ZIP.')
    target = Path(tempfile.mkdtemp(prefix='under_observation_'))
    with zipfile.ZipFile(archive) as bundle:
        for entry in bundle.infolist():
            resolved = (target/entry.filename).resolve()
            if not resolved.is_relative_to(target.resolve()):
                raise RuntimeError('Archive contains a path outside its destination.')
        bundle.extractall(target)
    ROOT = next(p.parent.parent.parent for p in target.rglob('data/source/game_metrics.csv'))
print('Project:', ROOT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)


## Behavioral comparison
Recompute source counts and the exploratory matched-seed bootstrap. These do not rerun the lab statement scorer.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'analysis/behavior.py')], check=True)


## Interpretable surrogate
Fit the fixed depth-three tree, hold out whole board seeds, and calculate permutation importance. Its target is tag presence, not truth or intentional deception.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'analysis/communication.py')], check=True)
results = json.loads((ROOT/'results/communication_surrogate.json').read_text())
print('Held-out AUROC:', results['out_of_fold_auc'])
print('Without audit status:', results['without_audit_feature_auc'])


In [ ]:
for row in sorted(results['importance'], key=lambda r: r['mean_auc_drop'], reverse=True):
    print(row['label'], round(row['mean_auc_drop'], 5))


## Rebuild and check
The HTML embeds the saved data, original logo, interface image and bundled fonts. The research tests check counts, source hashes and grouped validation.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'scripts/build_site.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT/'tests'), '-v'], check=True)


## Interpretation
The surrogate has limited fidelity. Its importance scores do not reveal Qwen’s internal reasoning. The oversight-effect intervals are inconclusive, and the note categories reproduce a supplied one-pass review. Read the website and documentation alongside these numerical outputs.